# Fine-tuning LoRA — Modèle médical expérimental

**Projet TechCorp AI Chat — Challenge IA (Ynov)** — mission expérimentale, mutualisée DATA/IA.

Ce notebook fine-tune **Phi-3.5-mini-instruct** (3.8B) en QLoRA sur le dataset médical
préparé par DATA (`ia/dataset/medical_dataset_sample_3000.json` dans le repo du groupe,
3000 exemples nettoyés depuis `ruslanmv/ai-medical-chatbot`).

⚠️ **Modèle expérimental, pas destiné à la production.** Voir les avertissements de
`medical_project/Readme.md` (dépôt source) : validation obligatoire par des
professionnels de santé avant tout usage réel.

Contexte sécurité : contrairement au modèle financier hérité (backdoor confirmée par
l'audit CYBER, voir `cyber/`), ce dataset médical est propre — vient directement de la
source HuggingFace publique, sans lien avec l'équipe précédente.

**Runtime requis** : GPU (menu *Exécution > Modifier le type d'exécution > T4 GPU*).

## 1. Installation des dépendances

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets trl

## 2. Récupération du dataset (depuis le repo du groupe)

In [ ]:
!git clone --depth 1 https://github.com/M1ck3y-ru/Rendu-Groupe-19.git

import json

DATASET_PATH = "Rendu-Groupe-19/ia/dataset/medical_dataset_sample_3000.json"

with open(DATASET_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"{len(raw_data)} exemples charges")
print(raw_data[0])

## 3. Chargement du modèle de base (4-bit, QLoRA)

`trust_remote_code=False` : Phi-3.5 est nativement supporté par `transformers`
récent, pas besoin d'exécuter du code distant (recommandation de l'audit CYBER,
finding F-ANSSI-R3, appliquée ici dès le départ plutôt que corrigée après coup).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

assert torch.cuda.is_available(), "Aucun GPU detecte - active le runtime GPU (Execution > Modifier le type d'execution)"

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=False,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Préparation du dataset (format chat Phi-3)

In [ ]:
from datasets import Dataset

def to_phi3_text(example):
    return {
        "text": f"<|user|>\n{example['instruction']}<|end|>\n<|assistant|>\n{example['output']}<|end|>"
    }

texts = [to_phi3_text(r) for r in raw_data]
hf_dataset = Dataset.from_list(texts)

# 90/10 train/eval pour suivre la loss de validation pendant l'entrainement
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"train: {len(train_dataset)}  eval: {len(eval_dataset)}")

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
eval_dataset = eval_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

## 5. Entraînement

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "./medical_lora_adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=True,
    report_to="none",
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
print(train_result.metrics)

## 6. Test rapide sur quelques questions médicales

In [ ]:
test_questions = [
    "I have had a persistent headache and blurred vision for two days, what could it be?",
    "What are the common side effects of ibuprofen?",
    "My child has a fever of 39C, when should I go to the ER?",
]

model.eval()
for q in test_questions:
    prompt = f"<|user|>\n{q}<|end|>\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, temperature=0.7, do_sample=True, top_p=0.9)
    response = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}\nA: {response}\n{'-'*60}")

## 7. À faire après l'entraînement

1. Relever dans les logs ci-dessus : loss finale (train + eval), nombre d'epochs, temps d'entraînement.
2. Reporter ces métriques dans `ia/rapport-finetuning-medical.md` du repo (section "Résultats du fine-tuning", à ajouter).
3. Partager le lien de ce notebook Colab dans le rendu (livrable demandé par `CONSIGNES.md`).
4. Optionnel : télécharger `medical_lora_adapter/` (bouton dossier à gauche dans Colab) si vous voulez conserver l'adaptateur — pas obligatoire pour le rendu, le modèle reste expérimental.